In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from PDF")
print("\n--- First page preview (first 500 chars)")
print(pages[0].page_content[:500])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600, # ~150 words per chunk
    chunk_overlap=100, # overlap keeps context at boundaries
    separators=["\n\n", "\n", ".", " "], # tries paragraph -> line -> sentence -> word
)

chunks = splitter.split_documents(pages)

len(chunks)

In [ ]:
chunks[0]

In [ ]:
len(chunks[0].page_content)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

test_query = "What is VolTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


# helper: join retrieved chunks into a single context string
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


SYSTEM_PROMPT = """\
You are a helpful telecom assistant
Answer the question using only the context provided below.
If the context does not content enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

# --- LLM via GROQ API
llm = ChatGroq(
    model = "qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed"
)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled.")

In [ ]:
question = "How does international roaming work and what charges should I expect?"

print(f"Q: {question}\n")
print(f"A: {chain.invoke(question)}")